# Module 03 — Notebook 01: GradCAM on ResNet-50

## Learning Objectives
By completing this notebook, you will:
1. Apply `LayerGradCam` to produce class activation maps for a pretrained ResNet-50
2. Compare GradCAM heatmaps across all four residual blocks (layer1–layer4)
3. Produce class-discriminative heatmaps for two different classes on the same image
4. Overlay GradCAM heatmaps on original images for communication-ready figures

## Prerequisites
- Module 02 Notebooks 01–03
- Guide 01 (GradCAM theory)

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["— Notebook 01: GradCAM on ResNet-50", "Apply `LayerGradCam` to produce class activation maps for a pretrained ResNet-50", "Compare GradCAM heatmaps across all four residual blocks (layer1–layer4)", "Produce class-discriminative heatmaps for two different classes on the same image", "Overlay GradCAM heatmaps on original images for communication-ready figures", "Module 02 Notebooks 01–03"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import urllib.request
import io
import json
from PIL import Image

from captum.attr import LayerGradCam, LayerAttribution

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

# Load pretrained ResNet-50
model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()
print("ResNet-50 loaded.")

In [ ]:
# Load ImageNet labels and images
LABELS_URL = (
    "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels"
    "/master/imagenet-simple-labels.json"
)
try:
    with urllib.request.urlopen(LABELS_URL) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]
    labels[208] = "Labrador retriever"
    labels[281] = "tabby cat"
    labels[463] = "bucket"

# Load a dog image
DOG_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
CAT_URL = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"

def load_image(url, fallback_class=208):
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            raw = Image.open(io.BytesIO(resp.read())).convert('RGB')
        return raw
    except Exception:
        # Fallback: solid-color image
        return Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))

dog_raw = load_image(DOG_URL)
cat_raw = load_image(CAT_URL)

dog_tensor = preprocess(dog_raw).unsqueeze(0).to(DEVICE)
cat_tensor = preprocess(cat_raw).unsqueeze(0).to(DEVICE)

dog_np = denormalize(dog_tensor.squeeze(0)).permute(1, 2, 0).numpy()
cat_np = denormalize(cat_tensor.squeeze(0)).permute(1, 2, 0).numpy()

with torch.no_grad():
    dog_class = model(dog_tensor).argmax().item()
    cat_class = model(cat_tensor).argmax().item()

print(f"Dog prediction: {labels[dog_class]} (class {dog_class})")
print(f"Cat prediction: {labels[cat_class]} (class {cat_class})")

## 1. GradCAM Across All Layers

We apply `LayerGradCam` to each of ResNet-50's four residual blocks. This shows the **hierarchical progression** of attention: early layers respond to local textures, late layers to semantic regions.

**ResNet-50 spatial resolutions:**
- `layer1`: 56×56 feature maps
- `layer2`: 28×28
- `layer3`: 14×14
- `layer4`: 7×7 (coarsest, most semantic)

In [ ]:
def compute_gradcam(model, input_tensor, target_layer, class_idx):
    """
    Compute GradCAM heatmap for a given layer and target class.

    Parameters
    ----------
    model : nn.Module
        Pretrained model (must be in eval mode)
    input_tensor : Tensor
        Input image, shape (1, 3, 224, 224)
    target_layer : nn.Module
        The layer to compute GradCAM at
    class_idx : int
        Target class index

    Returns
    -------
    heatmap : ndarray (224, 224)
        Normalized GradCAM heatmap, values in [0, 1]
    """
    lg = LayerGradCam(model, target_layer)

    # Compute attribution — shape (1, channels, H, W)
    attr = lg.attribute(input_tensor, target=class_idx)

    # Upsample to input resolution (224×224)
    attr_up = LayerAttribution.interpolate(
        attr, interpolate_dims=(224, 224), interpolate_mode='bilinear'
    )

    # Sum channels, apply ReLU, normalize
    heatmap = torch.relu(attr_up.sum(dim=1)).squeeze(0).detach().cpu()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    return heatmap.numpy()


# Define the four target layers
layer_dict = {
    'layer1\n(56×56)': model.layer1[-1],
    'layer2\n(28×28)': model.layer2[-1],
    'layer3\n(14×14)': model.layer3[-1],
    'layer4\n(7×7)':   model.layer4[-1],
}

# Compute GradCAM for dog image at each layer
print(f"Computing GradCAM for: {labels[dog_class]}")
dog_heatmaps = {}
for name, layer in layer_dict.items():
    dog_heatmaps[name] = compute_gradcam(model, dog_tensor, layer, dog_class)
    print(f"  {name.replace(chr(10), ' ')}: done")

print("\nGradCAM computed for all layers.")

In [ ]:
# Visualize GradCAM across layers
fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle(
    f'GradCAM Across All Layers — Prediction: "{labels[dog_class]}"',
    fontsize=13
)

# Column 0: original image
axes[0, 0].imshow(dog_np)
axes[0, 0].set_title('Original Image', fontsize=10)
axes[0, 0].axis('off')
axes[1, 0].imshow(dog_np)
axes[1, 0].set_title('Original Image', fontsize=10)
axes[1, 0].axis('off')

# Columns 1–4: GradCAM at each layer
for col, (name, heatmap) in enumerate(dog_heatmaps.items(), start=1):
    # Row 0: pure heatmap
    im = axes[0, col].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[0, col].set_title(f'GradCAM\n{name}', fontsize=9)
    axes[0, col].axis('off')
    plt.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Row 1: overlay on image
    axes[1, col].imshow(dog_np)
    axes[1, col].imshow(heatmap, alpha=0.55, cmap='jet', vmin=0, vmax=1)
    axes[1, col].set_title(f'Overlay\n{name}', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

print("""
Interpretation:
- layer1: diffuse, responds to edges/textures everywhere
- layer2: slightly more focused, responds to larger patterns
- layer3: concentrated on dog region, responds to object parts
- layer4: most concentrated, semantic dog region (ears, face, body)
""")

## 2. Class-Discriminative Heatmaps

GradCAM is **class-discriminative**: the same image produces different heatmaps for different target classes. This is one of GradCAM's advantages over class-agnostic methods like saliency.

We compute GradCAM for the top-5 predicted classes on the dog image and show how the heatmap changes.

In [ ]:
# Get top-5 predictions
with torch.no_grad():
    logits = model(dog_tensor)
    probs = torch.softmax(logits, dim=1)
    top5_probs, top5_classes = probs.topk(5, dim=1)

top5_probs = top5_probs.squeeze().tolist()
top5_classes = top5_classes.squeeze().tolist()

print("Top-5 predictions:")
for i, (cls, prob) in enumerate(zip(top5_classes, top5_probs)):
    print(f"  {i+1}. {labels[cls]:35s} ({prob:.1%})")

# Compute GradCAM for each of the top-5 classes
lg_layer4 = LayerGradCam(model, model.layer4[-1])

top5_heatmaps = []
for cls in top5_classes:
    h = compute_gradcam(model, dog_tensor, model.layer4[-1], cls)
    top5_heatmaps.append(h)

# Visualize
fig, axes = plt.subplots(2, 6, figsize=(24, 8))
fig.suptitle('Class-Discriminative GradCAM: Top-5 Classes (layer4)', fontsize=13)

# Column 0: original image
for row in range(2):
    axes[row, 0].imshow(dog_np)
    axes[row, 0].set_title('Input Image', fontsize=9)
    axes[row, 0].axis('off')

# Columns 1–5: per-class GradCAM
for col, (cls, prob, heatmap) in enumerate(
        zip(top5_classes, top5_probs, top5_heatmaps), start=1):
    short_label = labels[cls][:18] + ('...' if len(labels[cls]) > 18 else '')
    axes[0, col].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[0, col].set_title(f'#{col} {short_label}\n({prob:.1%})', fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(dog_np)
    axes[1, col].imshow(heatmap, alpha=0.55, cmap='jet', vmin=0, vmax=1)
    axes[1, col].set_title(f'#{col} Overlay', fontsize=8)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

## 3. GradCAM on Two Different Images

We now compare GradCAM for the dog and cat images side-by-side. Each model uses `layer4[-1]` as the target layer.

In [ ]:
# Compute GradCAM for cat image
cat_heatmap_layer4 = compute_gradcam(model, cat_tensor, model.layer4[-1], cat_class)
dog_heatmap_layer4 = dog_heatmaps['layer4\n(7×7)']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('GradCAM: Dog vs Cat — layer4 Heatmaps', fontsize=13)

rows = [
    (dog_np, dog_heatmap_layer4, f'Dog — {labels[dog_class]}'),
    (cat_np, cat_heatmap_layer4, f'Cat — {labels[cat_class]}'),
]

for row_idx, (img, heatmap, title) in enumerate(rows):
    # Original
    axes[row_idx, 0].imshow(img)
    axes[row_idx, 0].set_title('Original', fontsize=10)
    axes[row_idx, 0].axis('off')

    # GradCAM heatmap
    im = axes[row_idx, 1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[row_idx, 1].set_title(f'GradCAM\n{title}', fontsize=9)
    axes[row_idx, 1].axis('off')
    plt.colorbar(im, ax=axes[row_idx, 1], fraction=0.046)

    # Overlay
    axes[row_idx, 2].imshow(img)
    axes[row_idx, 2].imshow(heatmap, alpha=0.55, cmap='jet', vmin=0, vmax=1)
    axes[row_idx, 2].set_title('Overlay', fontsize=10)
    axes[row_idx, 2].axis('off')

    # Attribution histogram
    axes[row_idx, 3].hist(heatmap.flatten(), bins=50, color='steelblue',
                           edgecolor='none', alpha=0.8)
    axes[row_idx, 3].set_title(f'Attribution\nDistribution', fontsize=9)
    axes[row_idx, 3].set_xlabel('Heatmap value')
    axes[row_idx, 3].set_ylabel('Count')
    axes[row_idx, 3].axvline(0.5, color='red', linestyle='--',
                              label='Threshold=0.5')
    axes[row_idx, 3].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Quantify: what fraction of image area is "hot" (>0.5)?
dog_hot = (dog_heatmap_layer4 > 0.5).mean()
cat_hot = (cat_heatmap_layer4 > 0.5).mean()
print(f"Fraction of image with attribution > 0.5:")
print(f"  Dog: {dog_hot:.1%}")
print(f"  Cat: {cat_hot:.1%}")
print(f"A more concentrated heatmap (lower fraction) indicates sharper localization.")

## 4. Cross-Class Heatmap Comparison (Same Image)

A critical test: for the dog image, compute GradCAM for the **top dog class** and a **wrong class** (e.g., "tabby cat"). A well-trained model should produce different heatmaps for these two classes — the dog class should highlight dog features, the cat class should highlight different regions or nothing coherent.

In [ ]:
# Find class indices
# Labrador retriever ≈ 208, tabby cat ≈ 281 in ImageNet
dog_cls = dog_class
cat_cls = 281  # tabby cat

dog_on_dog_image = compute_gradcam(model, dog_tensor, model.layer4[-1], dog_cls)
cat_on_dog_image = compute_gradcam(model, dog_tensor, model.layer4[-1], cat_cls)

with torch.no_grad():
    dog_logits = model(dog_tensor)[0]
    dog_probs = torch.softmax(dog_logits, dim=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    f'Cross-Class GradCAM — Same Dog Image, Different Target Classes',
    fontsize=12
)

axes[0].imshow(dog_np)
axes[0].set_title('Input Image', fontsize=11)
axes[0].axis('off')

axes[1].imshow(dog_np)
axes[1].imshow(dog_on_dog_image, alpha=0.55, cmap='jet', vmin=0, vmax=1)
axes[1].set_title(
    f'Explaining: "{labels[dog_cls][:20]}"\n'
    f'Prob = {dog_probs[dog_cls]:.1%}',
    fontsize=10
)
axes[1].axis('off')

axes[2].imshow(dog_np)
axes[2].imshow(cat_on_dog_image, alpha=0.55, cmap='jet', vmin=0, vmax=1)
axes[2].set_title(
    f'Explaining: "{labels[cat_cls][:20]}"\n'
    f'Prob = {dog_probs[cat_cls]:.1%}',
    fontsize=10
)
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Measure spatial overlap between dog and cat class heatmaps
threshold = 0.5
dog_mask = (dog_on_dog_image > threshold)
cat_mask = (cat_on_dog_image > threshold)
overlap = (dog_mask & cat_mask).sum() / (dog_mask | cat_mask).sum().clip(1)

print(f"\nSpatial IoU (overlap) between dog-class and cat-class heatmaps: {overlap:.3f}")
print("Low IoU = class-discriminative model (different regions for different classes)")
print("High IoU = model using shared/background features")

## 5. Exercise: GradCAM on a New Image

Apply GradCAM to your own image and explore what the model attends to.

**Tasks:**
1. Load any image (or use the ones below)
2. Compute GradCAM at layer3 and layer4
3. Compute GradCAM for the top-1 and top-2 predicted classes
4. Report the spatial IoU between the two class heatmaps

**Starter code provided — fill in the `YOUR CODE HERE` sections.**

In [ ]:
# Exercise: GradCAM analysis on a new image

# Use the cat image (already loaded) or change to any URL
exercise_tensor = cat_tensor
exercise_np     = cat_np

# Step 1: Get predictions
with torch.no_grad():
    ex_probs = torch.softmax(model(exercise_tensor), dim=1)[0]
    ex_top2 = ex_probs.topk(2)

top1_class = ex_top2.indices[0].item()
top2_class = ex_top2.indices[1].item()
top1_prob  = ex_top2.values[0].item()
top2_prob  = ex_top2.values[1].item()

print(f"Top-1: {labels[top1_class]} ({top1_prob:.1%})")
print(f"Top-2: {labels[top2_class]} ({top2_prob:.1%})")

# Step 2: Compute GradCAM at layer3 and layer4 for top-1 class
heatmap_layer3 = compute_gradcam(model, exercise_tensor, model.layer3[-1], top1_class)
heatmap_layer4 = compute_gradcam(model, exercise_tensor, model.layer4[-1], top1_class)

# Step 3: Compute GradCAM for top-2 class at layer4
heatmap_top2 = compute_gradcam(model, exercise_tensor, model.layer4[-1], top2_class)

# Step 4: Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle(f'Exercise: GradCAM Analysis — {labels[top1_class]}', fontsize=12)

axes[0].imshow(exercise_np)
axes[0].set_title('Input')
axes[0].axis('off')

for ax, hm, ttl in [
    (axes[1], heatmap_layer3, f'layer3 / {labels[top1_class][:15]}'),
    (axes[2], heatmap_layer4, f'layer4 / {labels[top1_class][:15]}'),
    (axes[3], heatmap_top2,  f'layer4 / {labels[top2_class][:15]}'),
]:
    ax.imshow(exercise_np)
    ax.imshow(hm, alpha=0.55, cmap='jet', vmin=0, vmax=1)
    ax.set_title(ttl, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Step 5: Compute IoU between top-1 and top-2 heatmaps
mask1 = (heatmap_layer4 > 0.5)
mask2 = (heatmap_top2  > 0.5)
iou = (mask1 & mask2).sum() / max((mask1 | mask2).sum(), 1)

print(f"\nSpatial IoU (top-1 vs top-2 class heatmaps): {iou:.3f}")
print("Interpretation:")
if iou < 0.3:
    print("  Low IoU — model is attending to different regions for these two classes.")
    print("  Strong class-discriminative attribution.")
elif iou < 0.6:
    print("  Moderate IoU — some overlap in attended regions.")
    print("  The classes may share features (similar objects, backgrounds, textures).")
else:
    print("  High IoU — the two class heatmaps are nearly identical.")
    print("  The model may be using background or texture shortcuts.")

# Auto-check: verify heatmaps are valid
assert heatmap_layer4.min() >= 0.0, "Heatmap values should be non-negative"
assert heatmap_layer4.max() <= 1.0, "Heatmap values should be <= 1.0"
assert heatmap_layer4.shape == (224, 224), "Heatmap should be (224, 224)"
print("\nAll checks passed.")

## Summary

### What You Learned

1. **LayerGradCam** in Captum computes GradCAM for any convolutional layer
2. **Layer hierarchy:** early layers (layer1) are diffuse; final layers (layer4) are class-discriminative
3. **Class-discriminativeness:** different target classes produce different heatmaps for the same image
4. **Spatial IoU** quantifies how much two class heatmaps overlap — a low IoU indicates a class-discriminative model
5. **Resolution limitation:** layer4 heatmaps are coarse (7×7 before upsampling); layer1 is finer but less semantic

### Key API
```python
from captum.attr import LayerGradCam, LayerAttribution

lg = LayerGradCam(model, model.layer4[-1])
attr = lg.attribute(input_tensor, target=class_idx)
attr_up = LayerAttribution.interpolate(attr, (224, 224), 'bilinear')
heatmap = torch.relu(attr_up.sum(1)).squeeze()
```

### What's Next

**Notebook 02:** Layer Conductance — quantitatively measuring which layers contribute most to a prediction, using the IG integral applied at intermediate layers.